In [38]:
import scanpy as scp
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sb
import scipy as sp
import pandas as pd
import bbknn
import igraph
import json

from parameters import *

import warnings
warnings.filterwarnings('ignore')

In [2]:
adata = scp.read("results/QC.h5ad")

# Causal connection

In [3]:
bbknn.bbknn(adata,use_rep="X_pca",batch_key="Time",n_pcs=PCA,metric=METRIC,neighbors_within_batch=KNN,use_annoy=False)

In [31]:
def make_edges(a,key_stages,key_clusters,stages):

    #Disconnect
    m = pd.DataFrame()

    m["In"] = adata.uns["neighbors"]["distances"].nonzero()[0]
    m["Out"] = adata.uns["neighbors"]["distances"].nonzero()[1]
    m["Distances"] = np.array(adata.uns["neighbors"]["distances"][m["In"],m["Out"]])[0]
    m["Connectivities"] = np.array(adata.uns["neighbors"]["connectivities"][m["In"],m["Out"]])[0]
    m["StageIn"] = adata.obs.loc[adata.obs.iloc[m["In"]].index,key_stages].values
    m["StageOut"] = adata.obs.loc[adata.obs.iloc[m["Out"]].index,key_stages].values
    m["ClusterIn"] = adata.obs.loc[adata.obs.iloc[m["In"]].index,key_clusters].values
    m["ClusterOut"] = adata.obs.loc[adata.obs.iloc[m["Out"]].index,key_clusters].values
    m["Stage"] = [i+"_"+j for i,j in zip(m["StageIn"],m["StageOut"])]
    m["Cluster"] = [i+"_"+j for i,j in zip(m["ClusterIn"],m["ClusterOut"])]

    pos = np.array(range(adata.shape[0]))
    for i,stageIn in enumerate(stages):
        for j,stageOut in enumerate(stages):

            o0 = [i-1,i+1]
            
            if j not in o0:
                
                m = m.loc[m.loc[:,"Stage"] != (stageIn+"_"+stageOut),:]
    
    m["StageIn"] = m["StageIn"].cat.remove_unused_categories()
    m["StageOut"] = m["StageOut"].cat.remove_unused_categories()
    m["ClusterIn"] = m["ClusterIn"].cat.remove_unused_categories()
    m["ClusterOut"] = m["ClusterOut"].cat.remove_unused_categories()
    
    m["Forward"] = False
    for i,stageIn in enumerate(stages[:-1]):
        for j,stageOut in enumerate(stages[i+1:]):

                m.loc[m.loc[:,"Stage"] == (stageIn+"_"+stageOut),"Forward"] = True
#     m["Stage"] = m["Stage"].cat.remove_unused_categories()

    return m
    
# distances_prunned = sp.sparse.csc_matrix((m["Distances"],(m["In"],m["Out"])))
# connectivities_prunned = sp.sparse.csc_matrix((m["Connectivities"],(m["In"],m["Out"])))

In [32]:
m = make_edges(adata,"Time","leiden",np.sort(adata.obs["Time"].unique()))

In [36]:
s = m[m["Forward"].values].groupby(["ClusterIn","ClusterOut"]).size().unstack()
s2 = m[np.invert(m["Forward"].values)].groupby(["ClusterOut","ClusterIn"]).size().unstack()

sIn = s.transpose()
sIn = sIn/sIn.sum(axis=0)
sIn = sIn.transpose()

sIn = sIn.fillna(0)

sOut = s2.transpose()
sOut = sOut/sOut.sum(axis=0)
sOut = sOut.transpose()

sOut = sOut.fillna(0).transpose()

# Making the database

In [87]:
def make_json_graph_database(adata,sIn,sOut,savefile,key_gene,key_cluster,key_stage,geneList,key_annotation):

    #Make graph
    ss = sOut.values+sIn.values>0
    g = igraph.Graph.Adjacency(ss)
    g.vs["cluster"] = sIn.index.values
    g.vs["annotation"] = [adata.obs.loc[adata.obs[key_cluster]==j["cluster"],key_annotation][0] for j in g.vs]
    g.es["weightForward"] = [sIn.iloc[i,j] for i,j in g.get_edgelist()]
    g.es["weightBackward"] = [sOut.iloc[i,j] for i,j in g.get_edgelist()]
    g.es["weightCompensatedForward"] = [sCompensatedIn.iloc[i,j] for i,j in g.get_edgelist()]

    data = []
    for i,n in enumerate(g.vs):

        stage_num = float(n["cluster"].split("hr")[0].split("_")[0])

        data_prop = {
            "id": "n_"+str(i),
            "cluster": n["cluster"],
            "annotation": n["annotation"],
            "stage": stage_num,
            "stageInt": int(stage_num*4-6.5*4)
          }

        for i in geneList[:]:
            data_prop[i] = float(adata[adata.obs[key_cluster]==n["cluster"],adata.var[key_gene]==i].X.mean())
            l = float(adata[adata.obs[key_cluster]==n["cluster"],adata.var[key_gene]==i].X.todense().std())
            if np.isnan(l):
                data_prop[i+"_std"] = 0
            else:
                data_prop[i+"_std"] = l

        node = {
          "data": data_prop}

        data.append(node)

    for i,n in enumerate(g.es):

        edge = {
          "data": {
            "id": "e_"+str(i),
            "source": "n_"+str(n.source),
            "target": "n_"+str(n.target),
            "weightForward": n["weightForward"],
            "weightBackward": n["weightBackward"],
            "weightCompensatedForward": n["weightCompensatedForward"]
          }}

        data.append(edge)

    if ".json" in savefile:
        with open(savefile, 'w') as outfile:
            json.dump(data, outfile)
    else:
        savefile += ".json"
        with open(savefile, 'w') as outfile:
            json.dump(data, outfile)       
            
    for i,stage in enumerate(adata.obs[key_stage].unique()):
    
        adataAux = scp.read("results/dimensional_reduction_"+stage+".h5ad")
        X = adataAux.obsm["X_umap"]

        data = {
            "x": [float(i) for i in X[:,0]],
            "y": [float(i) for i in X[:,1]],
          }

        data["cluster_None"] = [5 for k in adata[adata.obs[key_stage]==stage].obs[key_cluster]==i]
        for j in adata[adata.obs[key_stage]==stage].obs[key_cluster].unique():
            cluster = "cluster_"+j.split("_")[-1]
            l = [5*int(k)+5 for k in adata[adata.obs[key_stage]==stage].obs[key_cluster]==j]
            data[cluster] = l.copy()

        for j in geneList[:]:
            l = np.array(adata[:,adata.var[key_gene]==j].X.todense())
            if l.shape[1] == 0:
                data[j] = [0 for i in range(l.shape[0])]
            else:
                data[j] = [float(i) for i in l[:,0]]

        with open("./cytoscape/plots/UMAP_"+stage+".json", 'w') as outfile:
            json.dump(data, outfile)
            
    return

In [89]:
genes = pd.read_excel("Marker_genes_scRNAseq_Gx.xlsx",header=None)

In [94]:
make_json_graph_database(adata,sIn,sOut,"cytoscape/data.json","gene_name","leiden","Time",genes.values[:,0],"leiden")

In [96]:
genes.values[:,0]

array(['T', 'Nkx2-5', 'Tbx5', 'Kdr', 'Pecam1', 'Cdh5', 'Cd34', 'Ptprc',
       'Kit', 'Ly6a', 'Ly6e', 'Itga2b', 'Spn', 'Flt3', 'Fcgr3', 'Slamf1',
       'Cd48', 'Esam', 'Notch2', 'Cd93', 'Robo4', 'Ifitm1', 'Ifitm2',
       'Ifitm3', 'Ccn3', 'Etv6', 'Runx1', 'Gata2', 'Meis1', 'Hoxb4',
       'Hoxa9', 'Bmi1', 'Pcgf2', 'Hlf', 'Foxo3', 'Mecom', 'Cbfa2t3',
       'Mllt3', 'Gata1', 'Hbb-bh1', 'Hbb-y', 'Hbb-bs', 'Hbb-bt', 'Bcl11b',
       'Klf1', 'Gfi1b', 'Spi1', 'Cebpa', 'Ebf1', 'Pax5', 'Notch1', 'Tcf3',
       'Gata3', 'Cd24a'], dtype=object)